# Compare cl 
This document checks if every model has been 

In [ ]:
# ! rm *.csv

# from google.colab import files
# uploaded = files.upload()
%ls

rm: cannot remove '*.csv': No such file or directory


Saving all_2h.csv to all_2h.csv
all_2h.csv  sample_data/


## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.preprocessing import LabelEncoder
import random

In [ ]:
from os import listdir
from os.path import isfile

Importing the collected data

In [ ]:
cryptos = ["biance-coin", "bitcoin-cash", "bitcoin", "cardano", "chainlink", "ethereum", "litecoin", "ripple", "stellar"]

## Helper Methods

In [ ]:
def plot_time_series(predicted, true, n_training, filename):
  """
  Plot the time series
  """
  plt.figure(figsize=(8,6)) #plotting
  plt.axvline(x=n_training, color="#ffd166", linestyle='-') #size of the training set

  plt.plot(predicted, label='Predicted Price', color="#118ab2") #predicted plot
  plt.plot(true, label='True Price', color="#06d6a0") #actual plot

  plt.title('Time-Series Prediction', fontsize=16)
  plt.xlabel('Time', fontsize=14)
  plt.ylabel('Price', fontsize=14)

  plt.xlim(0)
  plt.legend()
  plt.show()
  #plt.savefig(filename) 

In [ ]:
def classify(predicted_best, true_best):
    df = pd.DataFrame([predicted_best, true_best], columns=["pred","true"])

    total_correct = (predicted_best == true_best).sum()
    return totalcorrect/len(predict)

## LSTM Model


### Model Definition

LSTM Class used: https://pytorch.org/docs/master/generated/torch.nn.LSTM.html#torch.nn.LSTM

Some tutorials: https://pytorch.org/tutorials/beginner/nlp/sequence_models_tutorial.html



In [ ]:
class LSTMCustom(nn.Module):
    def __init__(self, num_classes, input_size, hidden_size, num_layers, seq_length):
        super(LSTMCustom, self).__init__()
        self.num_classes = num_classes #number of classes
        self.num_layers = num_layers #number of layers
        self.input_size = input_size #input size
        self.hidden_size = hidden_size #hidden state
        self.seq_length = seq_length #sequence length

        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size,
                          num_layers=num_layers, batch_first=True)
        #self.fc_1 =  nn.Linear(hidden_size, 128) #fully connected 1
        #self.fc = nn.Linear(128, num_classes) #fully connected last layer

        #self.relu = nn.ReLU()

        self.fc = nn.Linear(hidden_size, num_classes)
        self.sm = nn.Softmax()
    
    def forward(self,x):
        h_0 = Variable(torch.zeros(self.num_layers, x.size(0), self.hidden_size)) #hidden state
        c_0 = Variable(torch.zeros(self.num_layers, x.size(0), self.hidden_size)) #internal state
        # Propagate input through LSTM
        output, (hn, cn) = self.lstm(x, (h_0, c_0)) #lstm with input, hidden, and internal state

        out = self.fc(output[:,-1,:])
        return out

### Model Parameters

In [ ]:
num_epochs = 2000 #1000 epochs
learning_rate = 0.01 #0.001 lr

input_size = 1 #number of features
hidden_size = 32 #number of features in hidden state
num_layers = 8 #number of stacked lstm layers

num_classes = len(cryptos)

look_back = 12

In [ ]:
criterion = torch.nn.MSELoss()  # mean-squared error for regression

## Modeling

### Data Processing

In [ ]:
mm = MinMaxScaler()
ss = StandardScaler()

le = LabelEncoder()

In [ ]:
def train_test_split_tensor(x, y):
  """
  Custom train/test splitting
  TODO: maybe shorten code by using sklearn.preprocessing.train_test_split
  """
  cutoff = round(x.shape[0] * 0.8)

  # split into train and test
  x_train = x[:cutoff, :]
  x_test = x[cutoff:,:]
  y_train = y[:cutoff, :]
  y_test = y[cutoff:, :]
  
  return x_train, x_test, y_train, y_test, cutoff

In [ ]:
def get_x_y(df):

  # split into x and y
  x = df.drop(["best_crypto", "best_diff"],axis=1)

  x_ss = ss.fit_transform(x)

  x_2d = []
  for index in range(len(x_ss) - look_back):
    x_2d.append(x_ss[index: index + look_back])
  x_2d = Variable(torch.Tensor(np.array(x_2d)))

  y = le.fit_transform(df["best_crypto"])

  y = nn.functional.one_hot(torch.tensor(y, dtype=torch.int64), len(cryptos))
  
  return x_2d, y

### Training

In [ ]:
def train(x_train, y_train, lstm):
  """
  Train the lstm
  """

  for epoch in range(num_epochs):
    outputs = lstm.forward(x_train) #forward pass
    optimizer.zero_grad() #caluclate the gradient, manually setting to 0
  
    # obtain the loss function
    loss = criterion(outputs, y_train)
  
    loss.backward()
    optimizer.step()

    if epoch % 250 == 0:
      print("Epoch: %d, loss: %1.5f" % (epoch, loss.item())) 

  return lstm, cutoff

In [ ]:
results = []


In [ ]:
df = pd.read_csv("all_2h.csv")

x_2d, y = get_x_y(df)
x_train, x_test, y_train, y_test, cutoff = train_test_split_tensor(x_2d, y)


In [ ]:
print(x_train.shape, x_test.shape)
print(y_train.shape, y_test.shape)


torch.Size([3494, 12, 675]) torch.Size([874, 12, 675])
torch.Size([3494, 9]) torch.Size([886, 9])


In [ ]:
input_size = x_train.size(2)
lstm = LSTMCustom(num_classes, input_size, hidden_size, num_layers, df.shape[1]) 
optimizer = torch.optim.Adam(lstm.parameters(), lr=learning_rate) 

lstm, cutoff = train(x_train.float(), y_train.float(), lstm)
train_predict = lstm(x_train) #forward pass

Epoch: 0, loss: 0.12359
Epoch: 250, loss: 0.02401
Epoch: 500, loss: 0.01089
Epoch: 750, loss: 0.01176
Epoch: 1000, loss: 0.00434
Epoch: 1250, loss: 0.00345
Epoch: 1500, loss: 0.01299
Epoch: 1750, loss: 0.00445


In [ ]:
# data_predict = train_predict.data.numpy() #numpy conversion
# data_predict = mm.inverse_transform(data_predict) #reverse transformation

# dataY_plot = y_train.data.numpy()

# accuracy = classify()
# print(accuracy)
# dataY_plot = mm.inverse_transform(dataY_plot)

# results.append([filename, num_epochs, learning_rate, accuracy])


In [ ]:
# plot_name = f"{crypto_name}.png"
# plot_time_series(data_predict, dataY_plot, cutoff, plot_name)

### Results

In [ ]:
preds = train_predict.argmax(axis=1)

y = torch.tensor(le.fit_transform(df["best_crypto"]))[:3494]
train_acc = (preds == y).sum()/3494

In [ ]:
preds = lstm(x_test).argmax(axis=1)

y = y_test.argmax(axis=1)[look_back:]
test_acc = (preds == y).sum()/len(preds)

In [ ]:
print("input size: ", input_size)
print("epochs: ", num_epochs)
print("learning rate: ", learning_rate)
print("hidden size: ", hidden_size)
print("look back: ", look_back)

input size:  675
epochs:  2000
learning rate:  0.01
hidden size:  32
look back:  12


In [ ]:
pd.DataFrame([[test_acc, train_acc]], columns=["test accuracy", "training accuracy"])

,test accuracy,training accuracy
0,tensor(0.4462),tensor(0.9379)
